# HDAData package documentation

This package serves as a high-dimensional data analysis tool for Python using bioinformatics data, as part of the DAT-M-HIGHDIM lab project.

Contents: 
- [How to install](#-How-to-install)
- [Load Dependencies](#-Load-Dependencies)
- [Tutorial](#-Tutorial)

    The tutorial shows how to use the package's main functionalities:
    1. [Loading files and data structures](##1.-Loading-files-and-data-structures)
    2. [Distance functions for dynamic models and heatmaps](##2.-Distance-functions-for-dynamic-models-and-heatmaps)
    3. Clustering algorithms
    4. Dimensionality reduction algorithms

---
---

# How to install

1. Make sure you are in the root folder directory where `pyproject.toml` lives. 
2. In your terminal, install the package by running the following command:

    ```bash
    pip install -e .
    ```

# Load Dependencies

After installing the package, make sure to load the necessary dependencies first.

In [1]:
import myhda_ur as hda

# Tutorial

This tutorial shows how to use the HDAData package following the provided sample data in the `/data` folder as a guide.

<ins>Background on the sample data</ins>

<b><i>Scifate data</i></b>
- Sci-fate is a high-throughput sequencing method designed to track transcriptional dynamics of cells (i.e cells which are newly synthesized within a specific window of time) as compared to traditional methods which only observe a "static snapshot" of the cell at a specific moment
- By labelling new mRNA with the 4sU molecule, "old" and "new" genetic activity in the cell can be tracked over a 10-hour response
- Link to [original paper](https://www.nature.com/articles/s41587-020-0480-9) to learn more

<br>

| File name | Description | Information | Dataset size (gene x cell) | 
| --------- | ----------- | ------------ | ------------ |
matrix.mtx.gz | Gene expression matrix | Dataset showing the activity levels of all genes across all individual cells | (43750, 7404) |
4sU.Binom.ntr.mtx.gz | New to Total Ratio matrix | The proportion of a gene's total expression that is "new” | (43750, 7404) |
barcodes.tsv.gz | Barcodes + metadata | Unique molecular tags used to identify cells | (7404, 1) |
features.tsv.gz | Features + metadata | Metadata on genes measured | (43750, 5) |
trajectories_sci.csv | Trajectory table | Trajectory of how cells are connected to each other across time points | (949, 7) |


## 1. Loading files and data structures

This section of the tutorial shows you how to load the files, how to initialize the HDAData object, and work with the data structures.

- [Load files and Create HDAData Object](###-Load-files-and-Create-HDAData-Object)
- [Inspect object](###-Inspect-object)
- [View matrix](###-View-matrix)
- [Slicing](###-Slicing)
- [Gene gene expression](###-Get-gene-expression)

### Load files and Create HDAData Object

Firstly, we can load the scifate files and initialize an HDAData object, which comprises of the following elements:
- X = gene expression matrix
- obs = barcodes
- var = features
- layers = additional matrices (e.g new to total ratio matrix)
- similarities = similarities table
- trajectories = trajectory table

<i>Note that the HDAData object data structure is highly inspired by AnnData. </i>
> Source of inspiration: [AnnData](https://anndata.readthedocs.io/en/latest/tutorials/notebooks/getting-started.html)

In [2]:
# Option 1: Load the scifate data from the /data folder and initialize an HDAData Object directly

hdata = hda.load_scifate()

In [ ]:
# Option 2: Load with specific folder path and initialize the HDAData object manually 

loader = hda.Loader(dir = "<your_folder_path>")
loaded = loader.load_files()

hdata = hda.HDAData(
    X=loaded["gene_expression_matrix"],
    obs=loaded["barcodes_data"],
    var=loaded["features_data"],
    layers={"ntr": loaded["new_to_total_ratio_matrix"]},
    trajectories=loaded["trajectory_table"]
)

### Inspect object

In [ ]:
# Printing the HDAData object gives you an overview of its elements.

print(hdata)

HDAData object with:
  n_vars (genes): 43750
  n_obs (cells): 7404
  X shape: (43750, 7404)
  layers: ['ntr']
  trajectories: Yes


In [ ]:
# Inspect the Barcodes data (obs).

hdata.obs.head()

,name,cell,time,barcode
cell_name,,,,
A549.4h.AAAAACTCTCTCAA,A549.4h.AAAAACTCTCTCAA,A549,4h,AAAAACTCTCTCAA
A549.6h.AAAAAAGAAGTTCA,A549.6h.AAAAAAGAAGTTCA,A549,6h,AAAAAAGAAGTTCA
A549.8h.AAAAACGGAGAATA,A549.8h.AAAAACGGAGAATA,A549,8h,AAAAACGGAGAATA
A549.4h.AAAACGATCCTGGA,A549.4h.AAAACGATCCTGGA,A549,4h,AAAACGATCCTGGA
A549.6h.AAAAACTGAATTCC,A549.6h.AAAAACTGAATTCC,A549,6h,AAAAACTGAATTCC


In [ ]:
# Inspect the Features data (var).

hdata.var.head()

,gene_id,gene_symbol,feature_type,annotation,length
gene_symbol,,,,,
MTOR-AS1,ENSG00000225602,MTOR-AS1,Gene Expression,Exonic (h.ens90),705
ANGPTL7,ENSG00000171819,ANGPTL7,Gene Expression,Exonic (h.ens90),2238
RPL39P6,ENSG00000238173,RPL39P6,Gene Expression,Exonic (h.ens90),150
SNORA59A,ENSG00000239149,SNORA59A,Gene Expression,Exonic (h.ens90),152
AADACL4,ENSG00000204518,AADACL4,Gene Expression,Exonic (h.ens90),1575


In [ ]:
# Inspect the Trajectories table.

hdata.trajectories.head()

,0h,2h,4h,6h,8h,10h
1,A549.0h.CCGGGCGCTCGACG,A549.2h.GCTGGCGGTTATTG,A549.4h.CTACTGGTCCAGTC,A549.6h.GACAATGCTCGCAA,A549.8h.AAAAAACTACCGAC,A549.10h.AATGCAACCGGATC
2,A549.0h.CGTCCTTAGTTCAA,A549.2h.AGCTACTCCGCCAA,A549.4h.GTTACTCTTAGCGG,A549.6h.AGTCTCGTAGGCTT,A549.8h.AAAAACGGAGAATA,A549.10h.CTCCTGGATCAGGC
3,A549.0h.GGGATAGCTGCTAC,A549.2h.CAGGCGATCATTAC,A549.4h.CAAAGACGAGGTTG,A549.6h.CCTAGATCAGTCAT,A549.8h.AAAATAGAATAGCC,A549.10h.GGCTGACCTGGAAT
4,A549.0h.CGACATAATCTCGC,A549.2h.GCTAACTGGTAGAT,A549.4h.GTTCTCTCTCATCC,A549.6h.GACTGCTCATATGC,A549.8h.AAAATTCTACGCCA,A549.10h.CCGCCTCGGTCAAT
5,A549.0h.CTGTAGGTCTATGG,A549.2h.ATCTATCTAGGTTC,A549.4h.GCGGATCCTACCTC,A549.6h.GCGAGGCCTCCATT,A549.8h.AAACAACTACCGAC,A549.10h.ACCTGGTCCAGGAG


### View matrix

Display matrix (e.g X or ntr) in different formats by first n rows or columns. By default, it displays the gene expression matrix (X). 

Formats supported:
- COO Coordinate List format format (row, column, value) 
    -> returns None if it only contains zero entries
- Dense format -> numeric matrix in numpy arrays
- Labelled view -> labelled gene names as rows and cell names as columns in pandas DataFrame

In [ ]:
# Inspect the first 5 rows and columns of the gene expression matrix (X)

# Option 1: COO
print(f'First 5 rows and columns of X in COO format: ')
print(hdata.show(n_rows=5, n_cols=5, mode="coo"))
print(f'-'*20)

# Option 2: Dense
print(f'\nFirst 5 rows and columns of X in Dense format: ')
print(hdata.show(n_rows=5, n_cols=5, mode="dense"))
print(f'-'*20)

# Option 3: Labelled 
print(f'\nFirst 5 rows and columns of X in Label format: ')
df_X = hdata.show(n_rows=5, n_cols=5, mode="label")
df_X


First 5 rows and columns of X in COO format: 
None
--------------------

First 5 rows and columns of X in Dense format: 
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
--------------------

First 5 rows and columns of X in Label format: 


cell_name,A549.4h.AAAAACTCTCTCAA,A549.6h.AAAAAAGAAGTTCA,A549.8h.AAAAACGGAGAATA,A549.4h.AAAACGATCCTGGA,A549.6h.AAAAACTGAATTCC
gene_symbol,,,,,
MTOR-AS1,0,0,0,0,0
ANGPTL7,0,0,0,0,0
RPL39P6,0,0,0,0,0
SNORA59A,0,0,0,0,0
AADACL4,0,0,0,0,0


### Slicing

Get subset of HDAData using gene or cell names or index.

In [ ]:
# Example indexing patters:

hdata['Gene1', 'Cell1']                      # Single gene and cell
hdata[['Gene1', 'Gene2'], :]                 # Multiple genes, all cells
hdata[:, ['Cell1', 'Cell2']]                 # All genes, specific cells
hdata['Gene1':'Gene10', 'Cell1':'Cell5']     # Slice by name
hdata[:5, ['Cell1', 'Cell2']]                # Slice by gene index and cell names
hdata[['Gene1', 'Gene2'], :5]                # Slice by gene name and cell index
hdata[10:20, 30:40]                          # Integer indexing

In [ ]:
# Example 1: Get cell using cell name
cell1 = hdata[:, "A549.10h.GAGCTGAGACTCTA"]
cell1.show(n_rows=5, mode="label")

cell_name,A549.10h.GAGCTGAGACTCTA
gene_symbol,
MTOR-AS1,1
ANGPTL7,0
RPL39P6,0
SNORA59A,0
AADACL4,0


In [ ]:
# Example 2: Get gene using gene name
gene1 = hdata["MTOR-AS1", :]
gene1.show(n_cols=5, mode="label")

cell_name,A549.4h.AAAAACTCTCTCAA,A549.6h.AAAAAAGAAGTTCA,A549.8h.AAAAACGGAGAATA,A549.4h.AAAACGATCCTGGA,A549.6h.AAAAACTGAATTCC
gene_symbol,,,,,
MTOR-AS1,0,0,0,0,0


In [ ]:
# Example 3: Subset 10 rows and 10 columns from X
subset = hdata[10:20, 30:40]
subset.show(n_rows=5, n_cols=5, mode="label")

cell_name,A549.4h.AAAATCCGTCGCCA,A549.0h.AAAAACGCGTATCG,A549.2h.AAAACAGCAGTATC,A549.0h.AAAACATTATTACC,A549.10h.AAAACGTATTGAGA
gene_symbol,,,,,
PRAMEF12,0,0,0,0,0
PRAMEF20,0,0,0,0,0
AL354712.1,0,0,0,0,0
RPL7P11,0,0,0,0,0
AL357552.1,0,0,0,0,0


### Get gene expression

Query expression value directly for specific gene and cell.

In [ ]:
# Example 1: get gene expression using gene name and cell name
print(hdata.get_expression("MTOR-AS1", "A549.10h.GAGCTGAGACTCTA"))

# Example 2: get gene expression using gene index and cell index
print(hdata.get_expression(0, 0))

1


---

## 2. Distance functions for dynamic models and heatmaps

This section of the tutorial demonstrates the use of distance functions and heatmaps using an example biological use case on the scifate dataset.

<i><b>Biological Use Case: Identify dynamically regulated transcriptional programs during cortisol stress response using Highly Variable Genes (HVGs)</i></b>

The following similarities are explored:

| Analysis | Biological Interpretation |
| --------- | ----------- | 
| Gene-gene similarity | Co-regulated dynamic stress-response genes |
| Cell-cell similarity | Cells with similar transcriptional stress states |
| Trajectory similarity | Similar temporal response paths |

- [Highly Variable Genes (HVG)](###-Highly-Variable-Genes-(HVG))
- [Gene-gene similarity](###-Gene-gene-similarity)
- [Cell-cell similarity](###-Cell-cell-similarity)
- [Highly Dynamic Trajectories (HDT)](###-Highly-Dynamic-Trajectories-(HDT))
- [Trajectory-trajectory similarity](###-Trajectory-trajectory-similarity)

### Highly Variable Genes (HVG)

Since single-cell matrices are highly sparse, using all genes directly for analysis often produces unstable correlations. To obtain an informative subset from the sparse matrix, we first need to identify the Highly Variable Genes (HVG). 

In this tutorial, HVGs are identified using the NTR matrix because NTR captures active transcriptional dynamics, making it suitable for studying temporal stress responses. For each gene, the variance of its NTR values across all cells is computed. Genes with higher variance indicate more active transcriptional changes across cells, such as initiating stress response.

For a gene $g_i$, the variance across all cells is computed as:

$$
\mathbf{Var}(g_i) =
\frac{1}{N}\sum_{j=1}^{N}(NTR_{ij}-\mu_i)^2
$$

where:
- $NTR_{ij}$ is the NTR value of gene $i$ and cell $j$
- $N$ is the total number of cells
- $\mu_i$ is the mean NTR of gene $i$

The variance of the genes are then ranked decreasingly, where the top number of genes are selected as HVGs.

> Source of inspiration: [scanpy.experimental.pp.highly_variable_genes](https://scanpy.readthedocs.io/en/stable/generated/scanpy.experimental.pp.highly_variable_genes.html#scanpy.experimental.pp.highly_variable_genes) , 
[seurat highly variable genes](https://satijalab.org/seurat/articles/pbmc3k_tutorial.html)

In [3]:
# Get top 10 HVGs

top_10_genes = hdata.get_highly_variable_genes(n=10)
print(f'Top 10 gene names: {top_10_genes}')

Top 10 gene names: ['YOD1', 'SGK1', 'PARD6B', 'SMAD7', 'KLF6', 'KLF10', 'CEBPD', 'PLK2', 'ADM', 'GRPEL2']


In [4]:
top_10_genes = ['YOD1', 'SGK1', 'PARD6B', 'SMAD7', 'KLF6', 'KLF10', 'CEBPD', 'PLK2', 'ADM', 'GRPEL2']

In [4]:
# Initialize a temporary subset object for the top 10 HVGs to be used later

subset_top10 = hdata[top_10_genes, :]       # 10 genes, 7404 cells

### Gene-gene similarity 

After identifying the top 10 HVGs, we can then compare these genes whether they exhibit similar behaviour during stress response. For an intial approach, we can directly compare them using the raw NTR values. Each gene is represented by a vector of its NTR values across all cells:

$$
\mathbf{g}_i =
[NTR_{i1}, NTR_{i2}, ..., NTR_{iN}]
$$

where:
- $NTR_{ij}$ is the NTR value of HVG $i$ and cell $j$
- $N$ is the total number of cells

The distance between the vectors are then computed either by specifying the distance metric as:
- correlation, or
- cosine

From the distances, similarity scores between gene vectors can be computed into a resulting similarity matrix with diagonal entries:

$$
S = 
\begin{matrix}
S_{11} & S_{12} & \cdots & S_{1n} \\
S_{21} & S_{22} & \cdots & S_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
S_{n1} & S_{n2} & \cdots & S_{nn}
\end{matrix}
$$

where:
- $S_{ij}$ is similarity score of gene $i$ and gene $j$
- $n$ is the total number of HGV genes
- $S_{ii}$ = 1

The resulting similarity matrix is stored back into the original `hdata` object.

In [5]:
# Compute gene-gene similarity of top 10 HVGs with raw NTR

hdata.compute_similarity(
    data=subset_top10,
    key="ntr",
    kind="gene",
    method="correlation"
)

Similarity "correlation_ntr_gene" stored! 



HDAData object with:
  n_vars (genes): 43750
  n_obs (cells): 7404
  X shape: (43750, 7404)
  layers: ['ntr']
  similarities: ['correlation_ntr_gene']
  clusters: []
  trajectories: Yes

In [ ]:
# Inspect gene-gene similarity dataframe which was stored back into the original hdata object

sim1 = hdata.similarities["correlation_ntr_gene"]

,YOD1,SGK1,PARD6B,SMAD7,KLF6,KLF10,CEBPD,PLK2,ADM,GRPEL2
YOD1,1.000000,0.100134,0.062567,0.091714,0.169095,0.065951,0.067229,0.136405,0.122610,0.127108
SGK1,0.100134,1.000000,0.036117,0.083644,0.083344,0.053835,0.125137,0.057928,0.049556,0.077367
PARD6B,0.062567,0.036117,1.000000,0.054674,0.085799,0.064356,0.076837,0.049415,0.087599,0.095415
SMAD7,0.091714,0.083644,0.054674,1.000000,0.110295,0.089687,0.027949,0.090426,0.104398,0.093695
KLF6,0.169095,0.083344,0.085799,0.110295,1.000000,0.114303,0.077222,0.138555,0.169065,0.109297
KLF10,0.065951,0.053835,0.064356,0.089687,0.114303,1.000000,0.029685,0.106823,0.099798,0.073749
CEBPD,0.067229,0.125137,0.076837,0.027949,0.077222,0.029685,1.000000,0.006355,0.020968,0.037757
PLK2,0.136405,0.057928,0.049415,0.090426,0.138555,0.106823,0.006355,1.000000,0.192419,0.094190
ADM,0.122610,0.049556,0.087599,0.104398,0.169065,0.099798,0.020968,0.192419,1.000000,0.098048
GRPEL2,0.127108,0.077367,0.095415,0.093695,0.109297,0.073749,0.037757,0.094190,0.098048,1.000000


In [ ]:
# Plot clustered heatmap to visualize the similarity

hda.visualize.heatmap(
    sim=sim1, 
    type="gene", 
    cluster=True, 
    title="Gene-gene correlation similarity with raw NTR"
)

Now that we saw the similarity of top 10 HVGs using raw NTR values, we can investigate further by focusing on temporal behaviour rather than individual cell variability, in terms of transcriptional dynamics during cortisol stress response. 

To do so, we can utilize the information from the **trajectory matrix** by first constructing a temporal NTR matrix:

| HVG | 0h | .. | 10h |
| --- | --- | --- | --- |
| Gene 1 | mean NTR |  --- | mean NTR |
| Gene 2 | mean NTR |  --- | mean NTR |

where the mean NTR is computed from $NTR$[$HVG_{i}$, $cell_{j}$] for all cells at a timepoint in the trajectory matrix, given by:

$$
\mu_{i,t}
=
\frac{1}{N_t}
\sum_{j=1}^{N_t}
NTR_{ij}
$$

where:
- $t$ is a timepoint
- $N_{t}$ is the number of cells at timepoint $t$
- $NTR_{ij}$ is NTR value of HVG $i$ and cell $j$

Each gene row is now a temporal vector. As compared to using raw NTR previouly (10 HVG x 7404 cells), now we have (10 HVG x 7 temporal cells) which is less sparse and more interpretable.

In [9]:
# Compute gene-gene similarity of top 10 HVGs with mean NTR

hdata.compute_similarity(
    data=subset_top10,
    key="mean_ntr",
    kind="gene",
    method="correlation"
)

Similarity "correlation_mean_ntr_gene" stored! 



HDAData object with:
  n_vars (genes): 43750
  n_obs (cells): 7404
  X shape: (43750, 7404)
  layers: ['ntr']
  similarities: ['correlation_ntr_gene', 'correlation_mean_ntr_gene']
  clusters: []
  trajectories: Yes

In [ ]:
# Plot clustered heatmap to visualize the similarity

sim2 = hdata.similarities["correlation_mean_ntr_gene"]

hda.visualize.heatmap(
    sim=sim2, 
    type="gene", 
    cluster=True, 
    title="Gene-gene correlation similarity with temporal mean NTR"
)

### Cell-cell similarity

Next, we can analyze cell vs cell similarity to study cells with similar transcriptional cell states during cortisol stress response, based on the HVGs we identified.

To obtain a representative subset of cells, we randomly select a number of cells from each trajectory timepoint using the cell barcode metadata (obs).

Unlike the gene-gene analysis, which uses NTR values to study transcriptional dynamics, cell-cell similarity is computed using gene expression values, since it reflects the current transcriptional state of a cell.

A cell vector is defined as:

$$
\mathbf{c}_j =
[X_{1j}, X_{2j}, \ldots, X_{nj}]
$$

where:
- $X_{ij}$ is the gene expression value of HVG $i$ and cell $j$
- $n$ is the total number of HVG genes

Before computing distances, the cell vectors are normalized using shifted logarithm or log-transformation of normalized counts to stabalize variance across high and low gene expression values. The normalized vector $X'_{ij}$  is computed as:

$$
X'_{ij}
=
\log\left(
\frac{X_{ij}}
{\mathrm{library\_size}_j}
\times s
+1
\right)
$$

where:
- $library\_size{_j}$ is the total sum of count for all features for cell $j$
- $s$ is the scaling factor (e.g: 0.0001) to bring values to a comparable magnitude
- +1 is added to avoid log(0) for unexpressed genes

> Source of inspiration: [scanpy.pp.log1p](https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.log1p.html#scanpy.pp.log1p)

The normalized cell vectors are then used to compute cell-cell similarity using the same procedures as described in the gene-gene similarity section.

In [11]:
# Get two random cells for each timepoint

cells_hvg = hdata.get_timepoint_cells(n=2)
cells_hvg

['A549.0h.CGTTACGGCAACCA',
 'A549.0h.ACGCCTCGATGGAG',
 'A549.2h.GAGTCCTACGGCAA',
 'A549.2h.CCTTCCTAAGCGGT',
 'A549.4h.AGTTGTCGAGGTAT',
 'A549.4h.GACGATGAGTTCTC',
 'A549.6h.ATATTCCTTACCAA',
 'A549.6h.GCCTAATCGAACTC',
 'A549.8h.AAGGCGGCAGAGGA',
 'A549.8h.CCGAGACCGTAGCA',
 'A549.10h.GATTATCGTATCAA',
 'A549.10h.GAAAGACCTGGAAT']

In [12]:
# Initialize a temporary subset object for the selected cells to be used later

subset_hvg_cell = hdata[top_10_genes, cells_hvg]        # 10 genes, 12 cells

In [13]:
# Compute cell-cell similarity of the selected cells using normalized gene expression values 

hdata.compute_similarity(
    data=subset_hvg_cell,
    key="X",
    kind="cell",
    method="correlation"
)

Similarity "correlation_X_cell" stored! 



HDAData object with:
  n_vars (genes): 43750
  n_obs (cells): 7404
  X shape: (43750, 7404)
  layers: ['ntr']
  similarities: ['correlation_ntr_gene', 'correlation_mean_ntr_gene', 'correlation_X_cell']
  clusters: []
  trajectories: Yes

In [ ]:
# Plot clustered heatmap to visualize the similarity

sim3 = hdata.similarities["correlation_X_cell"]

hda.visualize.heatmap(
    sim=sim3, 
    type="cell", 
    cluster=True,
    title="Cell-cell correlation similarity"
)

### Highly Dynamic Trajectories (HDT)

As we have looked at gene-gene and cell-cell similarity, it is also possible to investigate trajectory-trajectory similarity. 

In order to identify a representative subset of trajectories to analyze, we can adopt a similar concept to the HVG, by retrieving the top Highly Dynamic Trajectories (HDT). 

Given that we have information about our HVG, for each trajectory, a matrix of shape (HVG $\times$ timepoints) is assembled using the NTR values of the HVG genes and cells at that timepoint, in order to capture the transcriptional dynamics of a trajectory over time. The matrix is then flattened row-wise into a single trajectory vector:

$$
\mathbf{traj}_i =
[NTR_{g1,0h}, NTR_{g1,2h}, ...,NTR_{g2,0h}, NTR_{g2,2h}, ...]
$$

where:
- $NTR_{g,t}$ is the NTR value of HVG $g$ and cell $j$ at timepoint $t$

We can then calculate the variance of each trajectory such that:

$$
\mathbf{Var}(traj_i) =
\frac{1}{M}\sum_{k=1}^{M}(v_{ik}-\bar{v})^2
$$

where:
- $v_{ik}$ is $k$-th element of the flattened vector
- $M$ is the total number of HVG $\times$ timepoints
- $\bar{v}$ is the mean of $traj_{i}$ 

The variance of the trajectories are then ranked decreasingly, where the top number of trajectories are selected as HDTs.

In [15]:
# Get top 10 HDTs

top_10_traj = subset_top10.get_highly_dynamic_trajectories(n=10)
top_10_traj

[884, 194, 157, 797, 926, 160, 252, 709, 349, 866]

### Trajectory-trajectory similarity

To investigate the similar temporal response path among trajectories during cortisol stress response, we can compute the trajectory-trajectory similarity using the selected HDTs.

We do the same by getting the flattened trajectory vectors ${traj}_i$ as described above, in order to compute the similarity matrix.

In [16]:
# Compute trajectory-trajectory similarity of HDTs using NTR values

hdata.compute_similarity(
    data=subset_top10,
    key="ntr",
    kind="trajectory",
    trajectory_id=top_10_traj,
    method="correlation"
)

Similarity "correlation_ntr_trajectory" stored! 



HDAData object with:
  n_vars (genes): 43750
  n_obs (cells): 7404
  X shape: (43750, 7404)
  layers: ['ntr']
  similarities: ['correlation_ntr_gene', 'correlation_mean_ntr_gene', 'correlation_X_cell', 'correlation_ntr_trajectory']
  clusters: []
  trajectories: Yes

In [ ]:
# Plot heatmap to visualize the similarity

sim4 = hdata.similarities["correlation_ntr_trajectory"]

hda.visualize.heatmap(
    sim=sim4, 
    type="trajectories", 
    cluster=True,
    title="Trajectory-trajectory correlation similarity"
)

---